# Episode 01: CDF and PMF with LeBron's season-debut FGM

Estimate an empirical distribution for LeBron James's 2026-27 season debut from his first regular-season appearance in each of the previous 21 completed seasons. FGM is discrete, so we use a PMF rather than a continuous PDF.

In [ ]:
from pathlib import Path
from time import sleep

import matplotlib.pyplot as plt
import pandas as pd
from nba_api.stats.endpoints import playergamelog

## 1. Define the sample

The seasons 2005-06 through 2025-26 give us 21 completed seasons before the 2026-27 target season.

In [ ]:
PLAYER_ID = 2544
START_YEAR = 2005
NUMBER_OF_SEASONS = 21
TARGET_SEASON = "2026-27"

def season_label(start_year):
    return f"{start_year}-{str(start_year + 1)[-2:]}"

SEASONS = [
    season_label(year)
    for year in range(START_YEAR, START_YEAR + NUMBER_OF_SEASONS)
]
SEASONS

## 2. Fetch the first appearance from each season

We sort before selecting the first game. This is LeBron's season debut, which may differ from his team's scheduled opening night.

In [ ]:
repo_root = Path.cwd().parent if Path.cwd().name == "python" else Path.cwd()
data_path = repo_root / "data" / "lebron_season_debut_fgm_2005_2025.csv"

def fetch_season_debuts():
    rows = []

    for season in SEASONS:
        games = playergamelog.PlayerGameLog(
            player_id=PLAYER_ID,
            season=season,
            season_type_all_star="Regular Season",
            timeout=60,
        ).get_data_frames()[0]

        if games.empty:
            raise RuntimeError(f"No regular-season games returned for {season}")

        games["GAME_DATE"] = pd.to_datetime(
            games["GAME_DATE"], format="mixed"
        )
        first_game = games.sort_values("GAME_DATE").iloc[0]

        rows.append({
            "season": season,
            "game_date": first_game["GAME_DATE"],
            "matchup": first_game["MATCHUP"],
            "fgm": int(first_game["FGM"]),
        })
        sleep(0.7)

    result = pd.DataFrame(rows).sort_values("game_date").reset_index(drop=True)
    data_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(data_path, index=False)
    return result

In [ ]:
if data_path.exists():
    opening_games = pd.read_csv(data_path, parse_dates=["game_date"])
else:
    opening_games = fetch_season_debuts()

assert len(opening_games) == NUMBER_OF_SEASONS
opening_games

## 3. Estimate the PMF and CDF

The PMF estimates $P(X=k)$. The CDF is the cumulative sum and estimates $P(X\leq k)$.

In [ ]:
fgm = opening_games["fgm"]
support = pd.Index(
    range(int(fgm.min()), int(fgm.max()) + 1),
    name="field_goals_made",
)

pmf = (
    fgm.value_counts(normalize=True)
    .sort_index()
    .reindex(support, fill_value=0.0)
)
cdf = pmf.cumsum()

distribution = pd.DataFrame({"pmf": pmf, "cdf": cdf})
distribution

## 4. Summarize the historical baseline

In [ ]:
threshold = 8
mean_fgm = fgm.mean()
mode_fgm = int(pmf.idxmax())
probability_at_most_8 = fgm.le(threshold).mean()

print(f"Empirical mean FGM: {mean_fgm:.2f}")
print(f"Empirical mode FGM: {mode_fgm}")
print(f"P(FGM <= 8): {probability_at_most_8:.1%}")

## 5. Visualize the empirical distribution

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(pmf.index, pmf.values, color="steelblue")
axes[0].set(
    title="Empirical PMF: LeBron Season-Debut FGM",
    xlabel="Field goals made",
    ylabel="Probability",
)

axes[1].step(cdf.index, cdf.values, where="post", color="darkorange")
axes[1].scatter(cdf.index, cdf.values, color="darkorange", s=24)
axes[1].set(
    title="Empirical CDF: P(FGM ≤ k)",
    xlabel="k field goals made",
    ylabel="Cumulative probability",
    ylim=(0, 1.05),
)

figure.suptitle(f"Historical baseline for the {TARGET_SEASON} season debut")
figure.tight_layout()
figure_path = repo_root / "videos" / "episode-01-pmf-cdf.png"
figure.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

## Interpretation and limitations

The PMF answers questions about exactly $k$ field goals. The CDF answers questions about at most $k$ field goals. This empirical distribution is a baseline, not a full forecast: it does not adjust for age, minutes, opponent, injury status, or role.